# 05. Agent Development Frameworks: Professional Implementations

This lab explores how modern agent frameworks encapsulate runtime mechanics. We will move beyond toy examples and examine professional practices: **dependency injection, strict typing, mock LLMs, and state reducers.**

**Scenario:** Northstar Commerce checkout failures have increased in Europe. We need an agent to investigate and recommend a safe action.

**Core Rule:** The framework is not the architecture. We must define the domain models and tool boundaries *first*.

## Part 1 — Domain Models & Tools
We use `pydantic` to enforce rigid structures and `logging` for observability, simulating a production environment.

In [1]:
from pydantic import BaseModel, Field
import json
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Northstar')

class RollbackProposal(BaseModel):
    deployment_id: str = Field(..., description='The deployment version to rollback')
    reason: str = Field(..., description='Technical justification')

def get_service_health(region: str) -> str:
    '''Returns the health status for a specific region.'''
    logger.info(f'Executing get_service_health for {region}')
    return json.dumps({'region': region, 'status': 'DEGRADED', 'error_rate': '15%'})

def get_recent_deployments(service: str) -> str:
    '''Returns the most recent deployment ID for a service.'''
    logger.info(f'Executing get_recent_deployments for {service}')
    return json.dumps({'service': service, 'latest_deployment_id': 'dep_eu_114', 'time': '2 hours ago'})

print('Domain objects and bounded tools successfully initialized.')

Domain objects and bounded tools successfully initialized.


## Part 2 — The Raw Agent Loop (Framework-Neutral)
To understand what frameworks actually do, we first build the loop manually. We implement a `MockLLM` that structurally mimics OpenAI's tool-calling API, allowing this code to run smoothly without credentials.

In [2]:
class MockLLM:
    def __init__(self):
        self.turns = 0
    def chat(self, messages: list) -> dict:
        self.turns += 1
        # Turn 1: LLM decides to use both tools
        if self.turns == 1:
            return {
                'role': 'assistant', 'content': None,
                'tool_calls': [
                    {'id': 'call_1', 'type': 'function', 'function': {'name': 'get_service_health', 'arguments': '{"region": "EU"}'}},
                    {'id': 'call_2', 'type': 'function', 'function': {'name': 'get_recent_deployments', 'arguments': '{"service": "checkout"}'}}
                ]
            }
        # Turn 2: LLM synthesizes the tool results
        return {
            'role': 'assistant',
            'content': 'The EU region is degraded due to deployment dep_eu_114. I recommend a rollback.',
            'tool_calls': None
        }

def raw_agent_loop(query: str):
    llm = MockLLM()
    messages = [{'role': 'user', 'content': query}]
    
    while True:
        response = llm.chat(messages)
        messages.append(response)
        
        if not response.get('tool_calls'):
            break  # Task complete
            
        for tc in response['tool_calls']:
            func_name = tc['function']['name']
            args = json.loads(tc['function']['arguments'])
            
            # The Application owns the routing and execution of tools
            if func_name == 'get_service_health':
                result = get_service_health(**args)
            elif func_name == 'get_recent_deployments':
                result = get_recent_deployments(**args)
            else:
                result = 'Error: Unknown tool'
                
            messages.append({'role': 'tool', 'tool_call_id': tc['id'], 'content': result})
            
    return messages[-1]['content']

final_output = raw_agent_loop('Checkout failures in EU. Investigate and recommend action.')
print('\nFinal Output:', final_output)

INFO: Executing get_service_health for EU


INFO: Executing get_recent_deployments for checkout



Final Output: The EU region is degraded due to deployment dep_eu_114. I recommend a rollback.


## Part 3 — PydanticAI: Typed Outputs & Dependency Injection
PydanticAI abstracts the loop away, but its primary value is **typing**. Notice how we explicitly inject dependencies (`RunContext`) and enforce a structured output.

In [3]:
import os
try:
    from pydantic_ai import Agent, RunContext
    from dataclasses import dataclass

    @dataclass
    class AppDependencies:
        db_connection_str: str
        environment: str

    class FinalDecision(BaseModel):
        requires_rollback: bool
        target_deployment: str
        confidence_score: float

    # The Agent enforces that the output matches FinalDecision
    typed_agent = Agent('openai:gpt-4o', deps_type=AppDependencies, result_type=FinalDecision)

    @typed_agent.tool
    def fetch_health(ctx: RunContext[AppDependencies], region: str) -> str:
        # Accessing injected context cleanly
        logger.info(f'[{ctx.deps.environment}] Querying via {ctx.deps.db_connection_str}')
        return get_service_health(region)

    print('PydanticAI architecture successfully defined.')
    
    if os.getenv('OPENAI_API_KEY'):
        # deps = AppDependencies(db_connection_str='postgres://app', environment='prod')
        # result = typed_agent.run_sync('Investigate EU.', deps=deps)
        pass
except ImportError:
    print('To run this framework, `pip install pydantic-ai`')


To run this framework, `pip install pydantic-ai`


## Part 4 — LangGraph: State Reducers & Interrupts
LangGraph views an agent as a state machine. Here we use the standard `messages` reducer pattern and demonstrate how to compile a graph with a forced interrupt for Human-in-the-Loop (HITL) approval.

In [4]:
try:
    from typing import Annotated
    from typing_extensions import TypedDict
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages

    # Standard Reducer: add_messages appends instead of overwriting
    class GraphState(TypedDict):
        messages: Annotated[list, add_messages]
        human_approved: bool

    def agent_node(state: GraphState):
        # In reality, this invokes the LLM
        logger.info('Node: Agent evaluating state.')
        return {'messages': [{'role': 'assistant', 'content': 'Proposing rollback.'}]}

    def review_node(state: GraphState):
        # A dedicated node for human verification
        logger.info('Node: Human review completed.')
        return {'human_approved': True}

    builder = StateGraph(GraphState)
    builder.add_node('agent', agent_node)
    builder.add_node('review', review_node)

    builder.add_edge(START, 'agent')
    builder.add_edge('agent', 'review')
    builder.add_edge('review', END)

    # We compile the graph with a breakpoint BEFORE the review node
    # This allows the system to sleep and wait for human input asynchronously.
    graph = builder.compile(interrupt_before=['review'])
    print('LangGraph state machine successfully compiled with explicit HITL interrupt.')
except ImportError:
    print('To run this framework, `pip install langgraph`')


LangGraph state machine successfully compiled with explicit HITL interrupt.


## Part 5 — OpenAI Agents SDK: Managed Tracing
The official SDK simplifies the loop while automatically generating rich traces for OpenAI observability.

In [5]:
try:
    from openai_agents import Agent, Runner

    incident_agent = Agent(
        name='IncidentCommander',
        instructions='Use tools to assess service health and recommend action.',
        tools=[get_service_health, get_recent_deployments]
    )
    print('OpenAI Agent instantiated.')

    if os.getenv('OPENAI_API_KEY'):
        print('OPENAI_API_KEY detected. Ready for execution.')
        # result = Runner.run_sync(incident_agent, 'Investigate EU checkout failures')
        # print(result.final_output)
except ImportError:
    print('To run this framework, `pip install openai-agents`')


To run this framework, `pip install openai-agents`


## Part 6 — Real API Execution (Optional)
If you have an API key, we will execute the raw loop logic using the real OpenAI client to demonstrate the latency, token usage, and dynamic tool selection.

In [6]:
if os.getenv('OPENAI_API_KEY'):
    try:
        from openai import OpenAI
        client = OpenAI()
        
        real_tools = [
            {'type': 'function', 'function': {'name': 'get_service_health', 'description': 'Check health for a region (e.g. EU)'}},
            {'type': 'function', 'function': {'name': 'get_recent_deployments', 'description': 'Check deployments for a service (e.g. checkout)'}}
        ]
        
        messages = [{'role': 'user', 'content': 'Checkout failures in EU. Investigate and recommend action.'}]
        print('Calling GPT-4o-mini...')
        
        # Execute one turn to see tool requests
        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=real_tools)
        msg = response.choices[0].message
        
        if msg.tool_calls:
            logger.info(f'Model requested {len(msg.tool_calls)} tool calls dynamically!')
            for tc in msg.tool_calls:
                logger.info(f'-> Requested Tool: {tc.function.name} with args {tc.function.arguments}')
        else:
            logger.info('Model did not request tools.')
            
    except ImportError:
        print('OpenAI client not installed.')
else:
    print('Skipping real execution. Set OPENAI_API_KEY to run.')

Skipping real execution. Set OPENAI_API_KEY to run.
